# NQ100 売買ルールバックテスト

Google Drive上のCSVを読み込み、東京時間の市場状態AIと売買シミュレーションを検証するためのColabノートブックです。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys
from pathlib import Path

# ColabでGitHubから開いた場合、必要に応じてリポジトリをcloneします。
REPO_DIR = Path('/content/NQ100')
if not REPO_DIR.exists():
    !git clone https://github.com/hon-daisuki/NQ100.git /content/NQ100

sys.path.insert(0, str(REPO_DIR))

In [ ]:
import pandas as pd
import numpy as np

from src.labels import add_future_return_label, add_direction_label
from src.backtest import build_long_short_returns, summarize_returns
from src.metrics import describe_time_range, missing_summary

## データ読み込み

In [ ]:
DATA_DIR = '/content/drive/MyDrive/CFD機械学習/backtest_ready'
CSV_NAME = 'USTEC_features_all.csv'

csv_path = Path(DATA_DIR) / CSV_NAME
df = pd.read_csv(csv_path)
df['日時'] = pd.to_datetime(df['日時'])
df = df.sort_values('日時').reset_index(drop=True)

describe_time_range(df)

In [ ]:
missing_summary(df)

## 東京時間フィルタとラベル作成

In [ ]:
df['hour'] = df['日時'].dt.hour
df_tokyo = df[(df['hour'] >= 8) & (df['hour'] <= 12)].copy()

df_tokyo = add_future_return_label(df_tokyo, horizon=4, threshold=0.005)
df_tokyo = add_direction_label(df_tokyo, return_col='future_return_4h', threshold=0.005)

df_tokyo[['日時', 'close', 'future_return_4h', 'is_trend', 'signal']].tail()

## 学習データ

In [ ]:
features_trend = [
    'return_1h', 'range', 'body', 'upper_wick', 'lower_wick',
    'hour', 'dayofweek', 'month', 'sma_5', 'sma_20',
    'macd', 'macd_signal', 'macd_hist', 'bb_width', 'rsi_14', 'atr_14'
]

df_model = df_tokyo.dropna(subset=features_trend + ['is_trend', 'future_return_4h']).copy()
split_index = int(len(df_model) * 0.8)

X_train = df_model[features_trend].iloc[:split_index]
X_test = df_model[features_trend].iloc[split_index:]
y_train = df_model['is_trend'].iloc[:split_index]
y_test = df_model['is_trend'].iloc[split_index:]

len(X_train), len(X_test), y_train.mean(), y_test.mean()

## LightGBM学習

In [ ]:
!pip -q install lightgbm

from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

model_trend = LGBMClassifier(
    n_estimators=500,
    learning_rate=0.03,
    num_leaves=31,
    class_weight='balanced',
    random_state=42,
)

model_trend.fit(X_train, y_train)
proba = model_trend.predict_proba(X_test)[:, 1]
pred = (proba >= 0.5).astype(int)

print('Accuracy =', accuracy_score(y_test, pred))
print(confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

## 簡易バックテスト

まずは「トレンド発生確率が高い時だけ、実際の未来リターン方向に基づく仮の方向シグナルで上限値を確認」します。次のステップで方向予測モデルに置き換えます。

In [ ]:
bt = df_model.iloc[split_index:].copy()
bt['trend_proba'] = proba
bt['trade_signal'] = np.where(bt['trend_proba'] >= 0.6, bt['signal'], 0)

returns = build_long_short_returns(
    bt,
    signal_col='trade_signal',
    return_col='future_return_4h',
    cost_per_trade=0.0002,
)

summarize_returns(returns)